# Notebook 06 — SPK Perankingan Prioritas Desa (TOPSIS vs SAW)

**Tujuan:** Menghasilkan **ranking prioritas desa/kelurahan** penerima intervensi bantuan
sosial di Kabupaten Lahat berdasarkan profil kemiskinan DTSEN (data 20 Juli 2026, 371 desa),
menggunakan dua metode SPK — **TOPSIS** dan **SAW** — lalu membandingkan keduanya.

**Posisi dalam penelitian:** Clustering (NB03/NB04) mengelompokkan desa menjadi *tipe*;
notebook ini memberi *urutan prioritas* di dalam & antar kelompok tersebut.

**Input:** `output/results/dtsen_clean.csv` (8 fitur proporsi) + label klaster K-Means.
**Output:** disimpan terpisah di `output/spk/`.

In [1]:
import sys
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt

ROOT = Path('../').resolve()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from utils import FEATURE_COLS, RESULTS_DIR

SPK_DIR = ROOT / 'output' / 'spk'
SPK_DIR.mkdir(parents=True, exist_ok=True)
matplotlib.rcParams.update({'figure.dpi': 150, 'font.size': 10})

# Data bersih + label klaster K-Means (untuk konteks)
df = pd.read_csv(RESULTS_DIR / 'dtsen_clean.csv')
km = pd.read_csv(RESULTS_DIR / 'hasil_kmeans.csv')[['KECAMATAN', 'KELURAHAN', 'cluster_kmeans']]
df = df.merge(km, on=['KECAMATAN', 'KELURAHAN'], how='left')

print(f'Jumlah desa : {len(df)}')
print(f'Fitur       : {FEATURE_COLS}')
df.head()

Jumlah desa : 371
Fitur       : ['pct_desil1', 'pct_desil2', 'pct_desil3', 'pct_desil4', 'pct_desil5', 'pct_desil6_10', 'pct_belum_prmk', 'pct_nonaktif']


,KECAMATAN,KELURAHAN,JUMLAH_KELUARGA,pct_desil1,pct_desil2,pct_desil3,pct_desil4,pct_desil5,pct_desil6_10,pct_belum_prmk,pct_nonaktif,cluster_kmeans
0,Tanjungsakti Pumu,Kembang Ayun,211,15.165877,9.004739,7.109005,13.270142,13.270142,39.810427,2.369668,5.213270,2
1,Tanjungsakti Pumu,Gunung Meraksa,457,17.286652,8.315098,6.345733,3.719912,23.632385,37.636761,3.063457,11.159737,0
2,Tanjungsakti Pumu,Tanjung Alam,456,13.377193,8.991228,5.701754,4.605263,17.982456,45.394737,3.947368,8.333333,1
3,Tanjungsakti Pumu,Batu Rancing,222,13.963964,11.711712,9.459459,7.657658,13.513514,42.342342,1.351351,6.756757,2
4,Tanjungsakti Pumu,Suban,346,15.895954,10.404624,9.248555,6.647399,19.653179,34.682081,3.468208,8.092486,0


## 1. Kriteria, Jenis, dan Bobot

Delapan fitur proporsi menjadi kriteria SPK. Kriteria *benefit* berarti "makin tinggi →
makin prioritas"; kriteria *cost* berarti sebaliknya (desa sejahtera → prioritas rendah).

> **Catatan:** bobot di bawah bersifat ILUSTRASI dan sebaiknya difinalkan melalui AHP
> atau wawancara dengan Dinas Sosial agar dapat dipertanggungjawabkan.

In [2]:
# CATATAN REVISI: kriteria 'pct_nonaktif' DIKELUARKAN dari pembobotan.
# Status nonaktif dapat terjadi karena sebab wajar (keluarga sudah mampu, pindah
# domisili, atau meninggal), sehingga tidak selalu menandakan kebutuhan intervensi.
# Berbeda dengan 'belum pemeringkatan' yang jelas berarti keluarga belum dapat
# dijangkau program bantuan karena belum memiliki peringkat desil.
KRITERIA = pd.DataFrame([
    ('pct_desil1',     'Benefit', 0.2778, '% Keluarga Desil 1 (termiskin)'),
    ('pct_desil2',     'Benefit', 0.2222, '% Keluarga Desil 2'),
    ('pct_desil3',     'Benefit', 0.1111, '% Keluarga Desil 3'),
    ('pct_desil4',     'Benefit', 0.0778, '% Keluarga Desil 4'),
    ('pct_desil5',     'Benefit', 0.0556, '% Keluarga Desil 5'),
    ('pct_desil6_10',  'Cost',    0.0889, '% Keluarga Desil 6-10 (sejahtera)'),
    ('pct_belum_prmk', 'Benefit', 0.1667, '% Keluarga belum diperingkat'),
], columns=['kode', 'jenis', 'bobot', 'keterangan'])

assert abs(KRITERIA['bobot'].sum() - 1.0) < 1e-3, 'Total bobot harus mendekati 1,0'
print('Jumlah kriteria:', len(KRITERIA), '(Nonaktif dikeluarkan)')
print('Total bobot:', round(KRITERIA['bobot'].sum(), 4))
KRITERIA

Jumlah kriteria: 7 (Nonaktif dikeluarkan)
Total bobot: 1.0001


,kode,jenis,bobot,keterangan
0,pct_desil1,Benefit,0.2778,% Keluarga Desil 1 (termiskin)
1,pct_desil2,Benefit,0.2222,% Keluarga Desil 2
2,pct_desil3,Benefit,0.1111,% Keluarga Desil 3
3,pct_desil4,Benefit,0.0778,% Keluarga Desil 4
4,pct_desil5,Benefit,0.0556,% Keluarga Desil 5
5,pct_desil6_10,Cost,0.0889,% Keluarga Desil 6-10 (sejahtera)
6,pct_belum_prmk,Benefit,0.1667,% Keluarga belum diperingkat


In [3]:
# Siapkan matriks keputusan
COLS   = KRITERIA['kode'].tolist()
W      = KRITERIA['bobot'].values
IS_BEN = (KRITERIA['jenis'] == 'Benefit').values          # True=benefit, False=cost
X      = df[COLS].values.astype(float)

print('Matriks keputusan:', X.shape, '(371 desa x 8 kriteria)')

Matriks keputusan: (371, 7) (371 desa x 8 kriteria)


## 2. Metode SAW (Simple Additive Weighting)

Langkah: (1) normalisasi rasio — *benefit*: `x / x_maks`, *cost*: `x_min / x`;
(2) skor = jumlah (bobot × nilai ternormalisasi). Skor tertinggi = prioritas tertinggi.

In [4]:
eps = 1e-9
R_saw = np.zeros_like(X)
for j in range(X.shape[1]):
    col = X[:, j]
    if IS_BEN[j]:
        R_saw[:, j] = col / (col.max() + eps)
    else:                                   # cost
        R_saw[:, j] = (col.min() + eps) / (col + eps)

skor_saw = (R_saw * W).sum(axis=1)
df['skor_saw'] = skor_saw.round(6)
df['rank_saw'] = df['skor_saw'].rank(ascending=False, method='min').astype(int)
print('SAW selesai. Rentang skor:', round(skor_saw.min(), 4), '-', round(skor_saw.max(), 4))

SAW selesai. Rentang skor: 0.1666 - 0.577


## 3. Metode TOPSIS

Langkah: (1) normalisasi vektor `x / akar(Σx²)`; (2) terbobot; (3) tentukan solusi ideal
positif A⁺ dan negatif A⁻; (4) hitung jarak Euclidean tiap desa ke A⁺ dan A⁻;
(5) skor kedekatan `C = D⁻ / (D⁺ + D⁻)`. Skor mendekati 1 = prioritas tertinggi.

In [5]:
# 1. normalisasi vektor
R_top = X / (np.sqrt((X**2).sum(axis=0)) + eps)
# 2. terbobot
V = R_top * W
# 3. solusi ideal
A_pos = np.where(IS_BEN, V.max(axis=0), V.min(axis=0))
A_neg = np.where(IS_BEN, V.min(axis=0), V.max(axis=0))
# 4. jarak
D_pos = np.sqrt(((V - A_pos)**2).sum(axis=1))
D_neg = np.sqrt(((V - A_neg)**2).sum(axis=1))
# 5. skor kedekatan
skor_top = D_neg / (D_pos + D_neg + eps)

df['skor_topsis'] = skor_top.round(6)
df['rank_topsis'] = df['skor_topsis'].rank(ascending=False, method='min').astype(int)
print('TOPSIS selesai. Rentang skor:', round(skor_top.min(), 4), '-', round(skor_top.max(), 4))

TOPSIS selesai. Rentang skor: 0.1331 - 0.5857


## 4. Perbandingan SAW vs TOPSIS

Dibandingkan dari tiga sisi: (a) korelasi peringkat (Spearman) — apakah kedua metode sepakat;
(b) kesepakatan pada 20 desa teratas; (c) daya pemisah skor (discrimination).

In [6]:
from scipy.stats import spearmanr

rho, _ = spearmanr(df['rank_saw'], df['rank_topsis'])

top20_saw = set(df.nsmallest(20, 'rank_saw')[['KECAMATAN', 'KELURAHAN']].apply(tuple, axis=1))
top20_top = set(df.nsmallest(20, 'rank_topsis')[['KECAMATAN', 'KELURAHAN']].apply(tuple, axis=1))
overlap20 = len(top20_saw & top20_top)

# daya pemisah: rasio simpangan baku terhadap rata-rata skor
cv_saw = df['skor_saw'].std() / df['skor_saw'].mean()
cv_top = df['skor_topsis'].std() / df['skor_topsis'].mean()

print('='*56)
print('  PERBANDINGAN SAW vs TOPSIS')
print('='*56)
print(f'  Korelasi Spearman peringkat   : {rho:.4f}')
print(f'  Kesepakatan 20 desa teratas   : {overlap20}/20')
print(f'  Daya pemisah SAW    (CV skor) : {cv_saw:.4f}')
print(f'  Daya pemisah TOPSIS (CV skor) : {cv_top:.4f}')
print('='*56)

  PERBANDINGAN SAW vs TOPSIS
  Korelasi Spearman peringkat   : 0.9479
  Kesepakatan 20 desa teratas   : 13/20
  Daya pemisah SAW    (CV skor) : 0.2272
  Daya pemisah TOPSIS (CV skor) : 0.2733


In [7]:
# Tabel 10 desa prioritas teratas menurut kedua metode
show = ['KECAMATAN', 'KELURAHAN', 'cluster_kmeans',
        'pct_desil1', 'pct_desil2', 'pct_belum_prmk',
        'skor_saw', 'rank_saw', 'skor_topsis', 'rank_topsis']
top10 = df.sort_values('rank_topsis').head(10)[show].round(4)
print('10 DESA PRIORITAS TERATAS (urut TOPSIS):')
print(top10.to_string(index=False))

10 DESA PRIORITAS TERATAS (urut TOPSIS):
         KECAMATAN         KELURAHAN  cluster_kmeans  pct_desil1  pct_desil2  pct_belum_prmk  skor_saw  rank_saw  skor_topsis  rank_topsis
       Muarapayang       Bandu Agung               0     39.4366     18.7793          4.2254    0.5633         2       0.5857            1
     Kikim Selatan  Beringin Janggut               0     44.3662     13.3803          4.9296    0.5366         7       0.5843            2
             Jarai       Lubuk Saung               0     30.5755     15.8273          5.3957    0.5168        11       0.5265            3
      Tanjungtebat     Padang Perigi               0     30.2905     19.9170          4.1494    0.5029        15       0.5156            4
Tanjung Sakti Pumi       Negeri Kaya               2     26.0000     26.6667          2.6667    0.5459         3       0.5035            5
     Kikim Selatan    Pulau Beringin               0     26.5795     20.6972          4.5752    0.5069        13       0.5003

In [8]:
# Scatter: peringkat SAW vs TOPSIS (makin dekat garis diagonal, makin sepakat)
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(df['rank_saw'], df['rank_topsis'], s=14, alpha=0.5, color='#2E86C1')
lim = [0, len(df) + 1]
ax.plot(lim, lim, 'r--', linewidth=1, label='Sepakat sempurna')
ax.set_xlabel('Peringkat SAW')
ax.set_ylabel('Peringkat TOPSIS')
ax.set_title(f'Kesepakatan Peringkat SAW vs TOPSIS\nSpearman = {rho:.3f}', fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig(SPK_DIR / 'perbandingan_rank_saw_topsis.png', dpi=150, bbox_inches='tight')
plt.close()
print('Gambar disimpan:', SPK_DIR / 'perbandingan_rank_saw_topsis.png')

Gambar disimpan: analysis\output\spk\perbandingan_rank_saw_topsis.png


In [9]:
# Simpan hasil lengkap
out = df[['KECAMATAN', 'KELURAHAN', 'cluster_kmeans'] + COLS +
         ['skor_saw', 'rank_saw', 'skor_topsis', 'rank_topsis']].sort_values('rank_topsis')
out.to_csv(SPK_DIR / 'ranking_prioritas_desa.csv', index=False)

import json
ringkas = {
    'jumlah_desa': int(len(df)),
    'metode': ['SAW', 'TOPSIS'],
    'korelasi_spearman': round(float(rho), 4),
    'kesepakatan_top20': f'{overlap20}/20',
    'cv_saw': round(float(cv_saw), 4),
    'cv_topsis': round(float(cv_top), 4),
    'catatan_bobot': 'Bobot ilustrasi; finalkan via AHP / wawancara Dinas Sosial.',
}
with open(SPK_DIR / 'ringkasan_spk.json', 'w', encoding='utf-8') as f:
    json.dump(ringkas, f, indent=2, ensure_ascii=False)
print('Hasil disimpan di:', SPK_DIR)
print(json.dumps(ringkas, indent=2, ensure_ascii=False))

Hasil disimpan di: analysis\output\spk
{
  "jumlah_desa": 371,
  "metode": [
    "SAW",
    "TOPSIS"
  ],
  "korelasi_spearman": 0.9479,
  "kesepakatan_top20": "13/20",
  "cv_saw": 0.2272,
  "cv_topsis": 0.2733,
  "catatan_bobot": "Bobot ilustrasi; finalkan via AHP / wawancara Dinas Sosial."
}


## 5. Ringkasan

- Kedua metode menghasilkan **skor prioritas** dan **peringkat** untuk 371 desa.
- Output: `output/spk/ranking_prioritas_desa.csv`, `perbandingan_rank_saw_topsis.png`,
  `ringkasan_spk.json`.
- Interpretasi mana yang lebih baik & kelebihan-kekurangan masing-masing diuraikan pada
  laporan (Bab IV), berdasarkan korelasi peringkat, kesepakatan top-20, dan daya pemisah.